# Baseline results: GPT-4.1, Gemini-2.5-flash-native, Landing AI

One summary table reports **overall correctness**, **completeness**, and **overall** score (averaged across documents) for:
- **GPT-4.1** (file-search free-form)
- **Gemini-2.5-flash-native** (file-search native JSON)
- **Landing AI** (ADE Parse + Extract)


In [1]:
import json
from pathlib import Path
import pandas as pd

# Resolve experiment-scripts (run from repo root or experiment-analysis/)
_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts. Run from repo root or experiment-analysis/.')

# Three baselines for reporting: GPT-4.1, Gemini-2.5-flash-native, Landing AI
BASELINE_RUNS = [
    # ('Gemini-2.5-on Landing AI Extracted Text', SCRIPT_BASE / 'baseline_landing_ai_w_gemini' / 'results' / 'gemini-2.5-flash'),
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
    # ('GPT-4.1', SCRIPT_BASE / 'baselines_file_search_results' / 'free_form' / 'gpt-4.1'),
]

def load_summary_metrics(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'summary_metrics.json'
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        o = data.get('overall', {})
        return (o.get('avg_correctness'), o.get('avg_completeness'), o.get('avg_overall'))
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    cols = data.get('columns', {})
    if not cols:
        return None
    c = [v.get('correctness', 0) for v in cols.values()]
    m = [v.get('completeness', 0) for v in cols.values()]
    o = [v.get('overall', 0) for v in cols.values()]
    return (sum(c) / len(c), sum(m) / len(m), sum(o) / len(o))

# Collect per-doc metrics for each model
common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        metrics = load_summary_metrics(run_path / pdf)
        if metrics is None:
            continue
        rows.append({
            'model': model_label,
            'pdf': pdf,
            'correctness': metrics[0],
            'completeness': metrics[1],
            'overall': metrics[2],
        })

df = pd.DataFrame(rows)
# One table: model x (Correctness, Completeness, Overall) — averages across docs
summary = df.groupby('model')[['correctness', 'completeness', 'overall']].mean()
summary = (summary * 100).round(2)
summary.columns = ['Correctness (%)', 'Completeness (%)', 'Overall (%)']
display(summary)


,Correctness (%),Completeness (%),Overall (%)
model,,,
Gemini-2.5-flash-native,82.48,80.75,81.62
Landing AI,86.73,87.37,87.05


### Per-document breakdown (Overall %)

Overall score by document for the same three baselines. AVG row = mean across documents.

In [2]:
# Per-document Overall (%) — same three baselines (uses df from cell above)
if 'df' in dir() and not df.empty:
    pivot = df.pivot(index='pdf', columns='model', values='overall')
    pivot = (pivot * 100).round(1)
    pivot.loc['AVG'] = pivot.mean(numeric_only=True)
    display(pivot)
else:
    print('Run the cell above first to build df.')


model,Gemini-2.5-flash-native,Landing AI
pdf,,
NCT00104715_Gravis_GETUG_EU'15,79.50,92.30
NCT00268476_Attard_STAMPEDE_Lancet'23,75.80,81.20
NCT00268476_James_STAMPEDE_IJC'22,83.30,86.80
NCT00309985_Kriayako_CHAARTED_JCO'18,95.50,90.40
NCT00309985_Sweeney_CHAARTED_NEJM'15,86.70,89.30
NCT01809691_Aggarwal_SWOG1216_JCO'22,78.20,91.40
NCT01957436_Fizazi_PEACE1_Lancet'22,72.40,83.30
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,79.90,88.70
NCT02799602_Hussain_ARASENS_JCO'23,75.90,83.60


### Numeric tolerance (correctness & completeness by document)

Per-document averages over columns in the **numeric_tolerance** category, for Gemini-2.5-flash-native and Landing AI only.


In [3]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts.')
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

def category_metrics(trial_dir: Path, category: str):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    cols = data.get('columns', {})
    subset = [v for v in cols.values() if v.get('category') == category]
    if not subset:
        return None
    c = sum(v.get('correctness', 0) for v in subset) / len(subset)
    m = sum(v.get('completeness', 0) for v in subset) / len(subset)
    return (c, m)

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        m = category_metrics(run_path / pdf, 'numeric_tolerance')
        if m is None:
            continue
        rows.append({'model': model_label, 'pdf': pdf, 'correctness': m[0], 'completeness': m[1]})
df_nt = pd.DataFrame(rows)
tab = df_nt.pivot_table(index='pdf', columns='model', values=['correctness', 'completeness'], aggfunc='mean')
tab = (tab * 100).round(2)
tab.columns = [f'{c[1]} {c[0].capitalize()} (%)' for c in tab.columns]
tab.loc['AVG'] = tab.mean(numeric_only=True)
display(tab)


,Gemini-2.5-flash-native Completeness (%),Landing AI Completeness (%),Gemini-2.5-flash-native Correctness (%),Landing AI Correctness (%)
pdf,,,,
NCT00104715_Gravis_GETUG_EU'15,76.640,94.390,80.370,95.330
NCT00268476_Attard_STAMPEDE_Lancet'23,72.900,84.580,79.910,84.580
NCT00268476_James_STAMPEDE_IJC'22,85.510,88.790,87.850,90.650
NCT00309985_Kriayako_CHAARTED_JCO'18,98.130,92.990,98.130,94.390
NCT00309985_Sweeney_CHAARTED_NEJM'15,90.650,92.520,90.650,92.520
NCT01809691_Aggarwal_SWOG1216_JCO'22,74.300,97.200,83.640,98.130
NCT01957436_Fizazi_PEACE1_Lancet'22,72.430,85.050,72.900,85.980
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,80.370,87.850,78.500,92.990
NCT02799602_Hussain_ARASENS_JCO'23,80.370,93.460,70.560,71.960


### Structured text (correctness & completeness by document)

Per-document averages over columns in the **structured_text** category, for Gemini-2.5-flash-native and Landing AI only.


In [4]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts.')
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

def category_metrics(trial_dir: Path, category: str):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    cols = data.get('columns', {})
    subset = [v for v in cols.values() if v.get('category') == category]
    if not subset:
        return None
    c = sum(v.get('correctness', 0) for v in subset) / len(subset)
    m = sum(v.get('completeness', 0) for v in subset) / len(subset)
    return (c, m)

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        m = category_metrics(run_path / pdf, 'structured_text')
        if m is None:
            continue
        rows.append({'model': model_label, 'pdf': pdf, 'correctness': m[0], 'completeness': m[1]})
df_st = pd.DataFrame(rows)
tab = df_st.pivot_table(index='pdf', columns='model', values=['correctness', 'completeness'], aggfunc='mean')
tab = (tab * 100).round(2)
tab.columns = [f'{c[1]} {c[0].capitalize()} (%)' for c in tab.columns]
tab.loc['AVG'] = tab.mean(numeric_only=True)
display(tab)


,Gemini-2.5-flash-native Completeness (%),Landing AI Completeness (%),Gemini-2.5-flash-native Correctness (%),Landing AI Correctness (%)
pdf,,,,
NCT00104715_Gravis_GETUG_EU'15,90.000,73.330,90.000,76.670
NCT00268476_Attard_STAMPEDE_Lancet'23,76.670,83.330,83.330,83.330
NCT00268476_James_STAMPEDE_IJC'22,66.670,76.670,66.670,76.670
NCT00309985_Kriayako_CHAARTED_JCO'18,80.000,80.000,80.000,80.000
NCT00309985_Sweeney_CHAARTED_NEJM'15,73.330,56.670,76.670,60.000
NCT01809691_Aggarwal_SWOG1216_JCO'22,83.330,80.000,83.330,80.000
NCT01957436_Fizazi_PEACE1_Lancet'22,63.330,63.330,63.330,60.000
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,73.330,73.330,76.670,76.670
NCT02799602_Hussain_ARASENS_JCO'23,66.670,83.330,70.000,86.670


### Hallucination count by document (numeric_tolerance only)

For each document and each baseline (Gemini-2.5-flash-native, Landing AI): **absolute count** of **numeric_tolerance** columns where ground truth had no value but the model predicted a value (hallucination). Only numeric columns are counted; exact_match and structured_text are excluded.

In [5]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts.')
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

def is_empty(val) -> bool:
    if val is None:
        return True
    s = str(val).strip().lower().strip(" .,:;")
    return s in {
        "", "nan", "not applicable", "not reported", "not present",
        "na", "n/a", "not found", "not available", "unknown",
    }

def hallucination_count_numeric_only(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    cols = data.get('columns', {})
    count = 0
    for _, col_data in cols.items():
        if col_data.get('category') != 'numeric_tolerance':
            continue
        gt = col_data.get('ground_truth', '')
        pred = col_data.get('predicted', '')
        if is_empty(gt) and not is_empty(pred):
            count += 1
    return count

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        n = hallucination_count_numeric_only(run_path / pdf)
        if n is None:
            continue
        rows.append({'model': model_label, 'pdf': pdf, 'hallucination_count': n})

df_h = pd.DataFrame(rows)
tab = df_h.pivot(index='pdf', columns='model', values='hallucination_count')
tab = tab.astype(int)
tab.loc['Total'] = tab.sum(numeric_only=True)
display(tab)

model,Gemini-2.5-flash-native,Landing AI
pdf,,
NCT00104715_Gravis_GETUG_EU'15,14,0
NCT00268476_Attard_STAMPEDE_Lancet'23,10,9
NCT00268476_James_STAMPEDE_IJC'22,0,6
NCT00309985_Kriayako_CHAARTED_JCO'18,2,2
NCT00309985_Sweeney_CHAARTED_NEJM'15,2,3
NCT01809691_Aggarwal_SWOG1216_JCO'22,2,0
NCT01957436_Fizazi_PEACE1_Lancet'22,14,5
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,4,4
NCT02799602_Hussain_ARASENS_JCO'23,1,4


In [6]:
# List all hallucinated columns (numeric_tolerance only: empty GT, non-empty pred) with GT and pred values per doc and model
import json
from pathlib import Path

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

def is_empty(val):
    if val is None:
        return True
    s = str(val).strip().lower().strip(" .,:;")
    return s in {"", "nan", "not applicable", "not reported", "not present", "na", "n/a", "not found", "not available", "unknown"}

def get_hallucinated_columns_numeric(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    out = []
    for col, col_data in data.get('columns', {}).items():
        if col_data.get('category') != 'numeric_tolerance':
            continue
        gt = col_data.get('ground_truth', '')
        pred = col_data.get('predicted', '')
        if is_empty(gt) and not is_empty(pred):
            out.append({"column": col, "ground_truth": gt if gt is not None else "", "predicted": pred})
    return out

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

# Structure: { "document_name": { "model_name": [ {"column": "...", "ground_truth": "", "predicted": "..."}, ... ] } }
halluc_by_doc = {}
for pdf in common_pdfs:
    halluc_by_doc[pdf] = {}
    for model_label, run_path in RUNS:
        if not run_path.exists():
            continue
        items = get_hallucinated_columns_numeric(run_path / pdf)
        halluc_by_doc[pdf][model_label] = items if items is not None else []

print(json.dumps(halluc_by_doc, indent=2, ensure_ascii=False))

{
  "NCT00104715_Gravis_GETUG_EU'15": {
    "Gemini-2.5-flash-native": [
      {
        "column": "Metastases - N (%) | Liver | Treatment",
        "ground_truth": "",
        "predicted": "28 (15%)"
      },
      {
        "column": "Metastases - N (%) | Liver | Control",
        "ground_truth": "",
        "predicted": "23 (12%)"
      },
      {
        "column": "OS Rate (%) | Overall | Treatment",
        "ground_truth": "",
        "predicted": "62.1 mo (95% confidence interval [CI], 49.5–73.7)"
      },
      {
        "column": "OS Rate (%) | Synchronous | Treatment",
        "ground_truth": "",
        "predicted": "52.6 mo (95% CI, 43.3-66.8)"
      },
      {
        "column": "OS Rate (%) | High volume | Control",
        "ground_truth": "",
        "predicted": "35.1 mo (95% CI, 29.9–43.6)"
      },
      {
        "column": "OS Rate (%) | Low volume | Control",
        "ground_truth": "",
        "predicted": "83.4 mo (95% CI, 61.8-NR)"
      },
      {
        "column"

### Original/Follow Up column — all trials

For each trial and each baseline: ground truth and predicted value for the **Original/Follow Up** column, and whether the model got it right (overall = 1.0) or wrong. Summary counts: how many correct vs incorrect per model.

In [7]:
import json
from pathlib import Path
import pandas as pd

COL = "Original/Follow Up"
_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for pdf in common_pdfs:
    for model_label, run_path in RUNS:
        if not run_path.exists():
            continue
        path = run_path / pdf / 'evaluation' / 'evaluation_results.json'
        if not path.exists():
            continue
        data = json.loads(path.read_text(encoding='utf-8'))
        cols = data.get('columns', {})
        if COL not in cols:
            rows.append({'document': pdf, 'model': model_label, 'ground_truth': None, 'predicted': None, 'correct': None})
            continue
        v = cols[COL]
        gt = v.get('ground_truth', '')
        pred = v.get('predicted', '')
        overall = v.get('overall', 0)
        rows.append({
            'document': pdf,
            'model': model_label,
            'ground_truth': gt if gt is not None else '',
            'predicted': pred if pred is not None else '',
            'correct': overall == 1.0,
        })

df = pd.DataFrame(rows)
display(df)

# Summary: how many right vs wrong per model
print("\n--- Summary (Original/Follow Up) ---")
for model in df['model'].unique():
    sub = df[df['model'] == model]
    valid = sub['correct'].notna()
    right = (sub.loc[valid, 'correct'] == True).sum()
    wrong = (sub.loc[valid, 'correct'] == False).sum()
    print(f"{model}: correct = {int(right)}, wrong = {int(wrong)}")

,document,model,ground_truth,predicted,correct
0,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,Follow-up / long-term OS analysis (median foll...,Follow up,True
1,NCT00104715_Gravis_GETUG_EU'15,Landing AI,Follow-up / long-term OS analysis (median foll...,Original,False
2,NCT00268476_Attard_STAMPEDE_Lancet'23,Gemini-2.5-flash-native,Follow Up,Follow Up,True
3,NCT00268476_Attard_STAMPEDE_Lancet'23,Landing AI,Follow Up,Original,False
4,NCT00268476_James_STAMPEDE_IJC'22,Gemini-2.5-flash-native,Follow Up,Follow Up,True
5,NCT00268476_James_STAMPEDE_IJC'22,Landing AI,Follow Up,Follow up,True
6,NCT00309985_Kriayako_CHAARTED_JCO'18,Gemini-2.5-flash-native,Follow-up / long-term survival analysis,Follow Up,True
7,NCT00309985_Kriayako_CHAARTED_JCO'18,Landing AI,Follow-up / long-term survival analysis,Follow up,True
8,NCT00309985_Sweeney_CHAARTED_NEJM'15,Gemini-2.5-flash-native,Original,Original,True
9,NCT00309985_Sweeney_CHAARTED_NEJM'15,Landing AI,Original,Original,True



--- Summary (Original/Follow Up) ---
Gemini-2.5-flash-native: correct = 10, wrong = 0
Landing AI: correct = 6, wrong = 4


### Region and Race columns — average accuracy by document

For each document and each model: **average overall score** (0–1) over all **Region - N (%)** columns and over all **Race - N (%)** columns. Table shows per-doc per-model averages; then overall mean across documents for each model.

In [8]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

def avg_overall_for_column_prefix(trial_dir: Path, prefix: str):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    cols = data.get('columns', {})
    scores = [v.get('overall', 0) for c, v in cols.items() if c.startswith(prefix)]
    if not scores:
        return None
    return sum(scores) / len(scores)

rows = []
for pdf in common_pdfs:
    for model_label, run_path in RUNS:
        if not run_path.exists():
            continue
        trial_dir = run_path / pdf
        avg_region = avg_overall_for_column_prefix(trial_dir, 'Region -')
        avg_race = avg_overall_for_column_prefix(trial_dir, 'Race -')
        rows.append({
            'document': pdf,
            'model': model_label,
            'avg_region': avg_region,
            'avg_race': avg_race,
        })

df = pd.DataFrame(rows)
# Show as percentages
df_display = df.copy()
df_display['avg_region (%)'] = (df_display['avg_region'] * 100).round(2)
df_display['avg_race (%)'] = (df_display['avg_race'] * 100).round(2)
display(df_display[['document', 'model', 'avg_region (%)', 'avg_race (%)']])

# Overall average per model (mean across documents)
print("\n--- Average accuracy (mean across documents) ---")
for model in df['model'].unique():
    sub = df[df['model'] == model]
    r = sub['avg_region'].dropna()
    c = sub['avg_race'].dropna()
    print(f"{model}: Region = {r.mean()*100:.2f}% (n={len(r)}), Race = {c.mean()*100:.2f}% (n={len(c)})")

,document,model,avg_region (%),avg_race (%)
0,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,58.33,100.00
1,NCT00104715_Gravis_GETUG_EU'15,Landing AI,83.33,100.00
2,NCT00268476_Attard_STAMPEDE_Lancet'23,Gemini-2.5-flash-native,83.33,100.00
3,NCT00268476_Attard_STAMPEDE_Lancet'23,Landing AI,83.33,100.00
4,NCT00268476_James_STAMPEDE_IJC'22,Gemini-2.5-flash-native,0.00,100.00
5,NCT00268476_James_STAMPEDE_IJC'22,Landing AI,83.33,85.71
6,NCT00309985_Kriayako_CHAARTED_JCO'18,Gemini-2.5-flash-native,100.00,100.00
7,NCT00309985_Kriayako_CHAARTED_JCO'18,Landing AI,100.00,100.00
8,NCT00309985_Sweeney_CHAARTED_NEJM'15,Gemini-2.5-flash-native,100.00,92.86
9,NCT00309985_Sweeney_CHAARTED_NEJM'15,Landing AI,100.00,100.00



--- Average accuracy (mean across documents) ---
Gemini-2.5-flash-native: Region = 79.58% (n=10), Race = 90.18% (n=10)
Landing AI: Region = 85.00% (n=10), Race = 93.93% (n=10)


In [9]:
# All Region and Race columns where both correctness and completeness are zero (with GT and predicted)
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for pdf in common_pdfs:
    for model_label, run_path in RUNS:
        if not run_path.exists():
            continue
        path = run_path / pdf / 'evaluation' / 'evaluation_results.json'
        if not path.exists():
            continue
        data = json.loads(path.read_text(encoding='utf-8'))
        for col, v in data.get('columns', {}).items():
            if not (col.startswith('Region -') or col.startswith('Race -')):
                continue
            c_val = v.get('correctness', 0)
            m_val = v.get('completeness', 0)
            if c_val != 0 or m_val != 0:
                continue
            rows.append({
                'document': pdf,
                'model': model_label,
                'column': col,
                'ground_truth': v.get('ground_truth', ''),
                'predicted': v.get('predicted', ''),
            })

df_zero = pd.DataFrame(rows)
# One row per (document, column) with both model outputs side by side
gt = df_zero.groupby(['document', 'column'])['ground_truth'].first().reset_index()
pred_pivot = df_zero.pivot_table(index=['document', 'column'], columns='model', values='predicted', aggfunc='first').reset_index()
merged = gt.merge(pred_pivot, on=['document', 'column'])
print(f"Total Region/Race columns with both scores zero: {len(df_zero)} (shown as {len(merged)} rows, one per doc+column)")
display(merged)

Total Region/Race columns with both scores zero: 52 (shown as 46 rows, one per doc+column)


,document,column,ground_truth,Gemini-2.5-flash-native,Landing AI
0,NCT00104715_Gravis_GETUG_EU'15,Region - N (%) | Africa | Control,,not found,NaN
1,NCT00104715_Gravis_GETUG_EU'15,Region - N (%) | Asia | Control,,not found,NaN
2,NCT00104715_Gravis_GETUG_EU'15,Region - N (%) | Asia | Treatment,,not found,NaN
3,NCT00104715_Gravis_GETUG_EU'15,Region - N (%) | Europe | Control,193 (100%),NaN,Not reported
4,NCT00104715_Gravis_GETUG_EU'15,Region - N (%) | Europe | Treatment,192 (100%),NaN,Not reported
5,NCT00104715_Gravis_GETUG_EU'15,Region - N (%) | North America | Treatment,,not found,NaN
6,NCT00104715_Gravis_GETUG_EU'15,Region - N (%) | Oceania | Control,,not found,NaN
7,NCT00268476_Attard_STAMPEDE_Lancet'23,Region - N (%) | Europe | Control,Abiraterone trial: 502 (100%) Abiraterone + En...,not found,Not reported
8,NCT00268476_Attard_STAMPEDE_Lancet'23,Region - N (%) | Europe | Treatment,Abiraterone trial: 501 (100%) Abiraterone + En...,not found,Not reported
9,NCT00268476_James_STAMPEDE_IJC'22,Race - N (%) | Other | Control,,NaN,26 (5%)


In [10]:
# Extract all columns from all PDFs for a method where correctness and completeness are not more than 1
# (i.e. filter: correctness <= 1 and completeness <= 1; set imperfect_only=True for at least one score < 1)
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
RUNS = [
    ('Gemini-2.5-flash-native', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Landing AI', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

# Option: set to a single (label, path) to extract for one method only
METHOD_FILTER = None  # None = all methods; or e.g. RUNS[0] for Gemini only

# Option: only include columns where at least one of correctness/completeness is < 1
imperfect_only = False

common_pdfs = None
for _, run_path in RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

runs_to_use = [METHOD_FILTER] if METHOD_FILTER is not None else RUNS
rows = []
for model_label, run_path in runs_to_use:
    if run_path is None or not run_path.exists():
        continue
    for pdf in common_pdfs:
        path = run_path / pdf / 'evaluation' / 'evaluation_results.json'
        if not path.exists():
            continue
        data = json.loads(path.read_text(encoding='utf-8'))
        for col, v in data.get('columns', {}).items():
            c_val = v.get('correctness', 0)
            m_val = v.get('completeness', 0)
            if c_val > 1 or m_val > 1:
                continue
            if imperfect_only and (c_val >= 1 and m_val >= 1):
                continue
            rows.append({
                'document': pdf,
                'method': model_label,
                'column': col,
                'ground_truth': v.get('ground_truth', ''),
                'predicted': v.get('predicted', ''),
                'correctness': c_val,
                'completeness': m_val,
                'overall': v.get('overall'),
                'category': v.get('category'),
            })

df_all_columns = pd.DataFrame(rows)
print(f"Total rows (doc × method × column with correctness, completeness <= 1): {len(df_all_columns)}")

# Write to JSON (list of records)
out_path = Path.cwd() / 'all_columns_filtered.json'
df_all_columns.to_json(out_path, orient='records', indent=2, date_format='iso')
print(f"Written to {out_path}")

display(df_all_columns)

Total rows (doc × method × column with correctness, completeness <= 1): 2660
Written to /mnt/data1/nahuja11/Mayo/CoRal-Map-Make/experiment-analysis/all_columns_filtered.json


,document,method,column,ground_truth,predicted,correctness,completeness,overall,category
0,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,Control Arm - N,193 (ADT alone),193,1.0,0.5,0.75,numeric_tolerance
1,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,Docetaxel administration - N (%) | Control,0 (0%),127 (65.8%),0.0,0.0,0.00,numeric_tolerance
2,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,Docetaxel administration - N (%) | Treatment,192 (100%),192 (100%),1.0,1.0,1.00,numeric_tolerance
3,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,Median Follow-Up Duration (mo),83.9 months,83.9 mo,1.0,1.0,1.00,numeric_tolerance
4,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,Median Age (years) | Control,64.0,64.0 (58.0-70.0),1.0,1.0,1.00,numeric_tolerance
...,...,...,...,...,...,...,...,...,...
2655,NCT02799602_Smith_ARASENS_NEJM'22,Landing AI,Control Arm,Placebo + ADT + docetaxel,Placebo + ADT + Docetaxel,1.0,1.0,1.00,structured_text
2656,NCT02799602_Smith_ARASENS_NEJM'22,Landing AI,Treatment Class,Androgen-receptor pathway inhibitor,Androgen-receptor inhibitor + ADT + Chemotherapy,0.5,0.5,0.50,structured_text
2657,NCT02799602_Smith_ARASENS_NEJM'22,Landing AI,Type of Therapy,Triplet therapy (ADT + docetaxel + AR inhibitor),"Combination therapy (darolutamide, ADT, doceta...",1.0,1.0,1.00,structured_text
2658,NCT02799602_Smith_ARASENS_NEJM'22,Landing AI,Trial,ARASENS,ARASENS,1.0,1.0,1.00,structured_text
